# Capítulo 7. Árboles de decisión

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Árboles de decisión para clasificación

La formulación matemática de **entropía, impureza, particiones y árboles de decisión** se desarrolla con mayor profundidad
en los capítulos 3, 6 y 13 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de explicar la idea básica de un árbol de decisión, entrenarlo en R y evaluar su desempeño.

## Introducción

Los árboles de decisión clasifican observaciones mediante una secuencia de preguntas. Son interpretables y pueden expresarse como reglas tipo si-entonces.

## Fundamento matemático

Una medida común de impureza es el índice de Gini:

$$
Gini = 1 - \sum_{k=1}^{K} p_k^2
$$

## Cargar paquetes y datos


In [ ]:
library(readr)
library(dplyr)
library(rpart)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  print("Base preparada cargada correctamente.")
} else {
  atus_ml <- NULL
  print("No se encontró el archivo datos/atus_ml_preparado.csv. Ejecuta primero el capítulo 2.")
}


## Usar la base completa


In [ ]:
if (!is.null(atus_ml)) {
  atus_arbol_base <- atus_ml |>
    mutate(
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños")),
      MES = as.factor(MES),
      ID_HORA = as.numeric(ID_HORA),
      DIASEMANA = as.factor(DIASEMANA),
      TIPACCID = as.factor(TIPACCID),
      CAUSAACCI = as.factor(CAUSAACCI)
    ) |>
    na.omit()

  data.frame(filas = nrow(atus_arbol_base), columnas = ncol(atus_arbol_base))
}


## Explicación del código
Se usa la base completa, pero el crecimiento del árbol se controla mediante parámetros del modelo.

## División y entrenamiento


In [ ]:
if (exists("atus_arbol_base")) {
  set.seed(123)
  idx <- sample(1:nrow(atus_arbol_base), size = round(0.7 * nrow(atus_arbol_base)))
  entrenamiento <- atus_arbol_base[idx, ]
  prueba <- atus_arbol_base[-idx, ]

  modelo_arbol <- rpart(
    accidente_con_victimas ~ MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
    data = entrenamiento,
    method = "class",
    control = rpart.control(cp = 0.005, minsplit = 1000, minbucket = 300, maxdepth = 5)
  )
}


## Explicación del código
Se ajusta un árbol de decisión controlado para evitar sobreajuste y salidas demasiado grandes.

## Complejidad e importancia


In [ ]:
if (exists("modelo_arbol")) printcp(modelo_arbol)

if (exists("modelo_arbol")) {
  importancia <- modelo_arbol$variable.importance
  if (!is.null(importancia)) {
    importancia_df <- data.frame(variable = names(importancia), importancia = as.numeric(importancia), row.names = NULL)
    importancia_df <- importancia_df[order(-importancia_df$importancia), ]
    importancia_df
  }
}


## Interpretación del resultado
La importancia de variables indica qué variables ayudan más a separar accidentes con víctimas de accidentes de solo daños.

## Gráfica opcional del árbol

La gráfica completa del árbol puede ser pesada en PDF. El lector puede ejecutarla manualmente en RStudio.


In [ ]:
plot(modelo_arbol, uniform = TRUE, margin = 0.1)
text(modelo_arbol, use.n = TRUE, cex = 0.7)


## Predicción y métricas


In [ ]:
if (exists("modelo_arbol")) {
  pred_arbol <- predict(modelo_arbol, newdata = prueba, type = "class")
  real_arbol <- factor(prueba$accidente_con_victimas, levels = c("Con víctimas", "Solo daños"))
  pred_arbol <- factor(pred_arbol, levels = c("Con víctimas", "Solo daños"))

  matriz_confusion_arbol <- table(Real = real_arbol, Predicho = pred_arbol)
  matriz_confusion_arbol
}

if (exists("matriz_confusion_arbol")) {
  VP <- matriz_confusion_arbol["Con víctimas", "Con víctimas"]
  FN <- matriz_confusion_arbol["Con víctimas", "Solo daños"]
  FP <- matriz_confusion_arbol["Solo daños", "Con víctimas"]
  VN <- matriz_confusion_arbol["Solo daños", "Solo daños"]

  data.frame(exactitud=(VP+VN)/(VP+FN+FP+VN), sensibilidad=VP/(VP+FN), especificidad=VN/(VN+FP))
}


## Interpretación del resultado
La matriz de confusión y las métricas permiten evaluar el desempeño del árbol sobre datos de prueba.

## Laboratorio interactivo: construye una división del árbol

Este laboratorio permite explorar cómo una regla del tipo
`Petal.Length < punto de corte` divide los datos de `iris`. El usuario puede
cambiar el punto de corte y comparar la impureza de Gini antes y después de la
división.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

La versión web permite cambiar la variable y el punto de corte, observar la
división de las especies de `iris` y comparar la impureza de Gini de los nodos.

## Caso aplicado B: árbol de decisión con COVID-19

Usamos la misma muestra común de hasta 12 000 registros de COVID-19 México 2022 y la misma semilla `2026`. El árbol permite observar reglas de decisión fáciles de interpretar.

> **Uso académico:** las reglas encontradas describen el comportamiento de esta muestra y no deben interpretarse como reglas clínicas.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_arbol <- read_csv(ruta_covid, show_col_types = FALSE) |>
    select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
           OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    tidyr::drop_na()

  set.seed(2026)
  covid_arbol <- covid_arbol |>
    sample_n(min(12000, nrow(covid_arbol))) |>
    mutate(MURIO = factor(MURIO, levels = c(1, 0), labels = c("Defunción", "Sin defunción")))

  set.seed(2026)
  idx_covid <- sample(seq_len(nrow(covid_arbol)), size = floor(0.80 * nrow(covid_arbol)))
  covid_train <- covid_arbol[idx_covid, ]
  covid_test <- covid_arbol[-idx_covid, ]
}


In [ ]:
if (exists("covid_train")) {
  arbol_covid <- rpart(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION + OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = covid_train,
    method = "class",
    control = rpart.control(cp = 0.002, minsplit = 80, minbucket = 30, maxdepth = 5)
  )
  printcp(arbol_covid)
}


In [ ]:
if (exists("arbol_covid")) {
  imp_covid_arbol <- arbol_covid$variable.importance
  if (!is.null(imp_covid_arbol)) {
    data.frame(variable = names(imp_covid_arbol), importancia = as.numeric(imp_covid_arbol)) |>
      arrange(desc(importancia))
  }
}


In [ ]:
if (exists("arbol_covid")) {
  pred_covid_arbol <- predict(arbol_covid, covid_test, type = "class")
  matriz_covid_arbol <- table(
    Real = factor(covid_test$MURIO, levels = c("Defunción", "Sin defunción")),
    Predicho = factor(pred_covid_arbol, levels = c("Defunción", "Sin defunción"))
  )
  matriz_covid_arbol

  VP <- matriz_covid_arbol[1,1]; FN <- matriz_covid_arbol[1,2]
  FP <- matriz_covid_arbol[2,1]; VN <- matriz_covid_arbol[2,2]

  data.frame(
    exactitud = (VP + VN) / sum(matriz_covid_arbol),
    sensibilidad = ifelse(VP + FN == 0, NA, VP / (VP + FN)),
    especificidad = ifelse(VN + FP == 0, NA, VN / (VN + FP))
  )
}


## Interpretación
La principal ventaja del árbol es que transforma el modelo en una secuencia visible de decisiones. Su desventaja es que un solo árbol puede cambiar bastante ante pequeñas variaciones de la muestra; esto motiva Random Forest.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=UperQjxBYEY) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-07/capitulo-07-arboles-presentacion.pdf) | [Descargar PDF](recursos/capitulo-07/capitulo-07-arboles-presentacion.pdf){download="capitulo-07-arboles-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-07/capitulo-07-arboles-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-07/capitulo-07-arboles-presentacion.pptx){download="capitulo-07-arboles-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-07/capitulo-07-arboles-infografia.png) | [Descargar PNG](recursos/capitulo-07/capitulo-07-arboles-infografia.png){download="capitulo-07-arboles-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/07-arboles-decision.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 7](recursos/capitulo-07/capitulo-07-arboles-infografia.png)](recursos/capitulo-07/capitulo-07-arboles-infografia.png)

**Video del capítulo:** <https://www.youtube.com/watch?v=UperQjxBYEY>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Conclusión

Los árboles de decisión son modelos intuitivos, visuales e interpretables. Este capítulo prepara el camino para modelos basados en muchos árboles, como Random Forest.

## Referencias fundamentales de los árboles de decisión

La metodología CART fue desarrollada sistemáticamente por @breiman1984classification. El algoritmo ID3 y la inducción de árboles mediante ganancia de información fueron presentados por @quinlan1986induction. Para una introducción contemporánea con ejemplos en R puede consultarse @james2021islr.
